# Aarogya — Fine-tune Gemma 4 4B on Rural Health Triage

**Hackathon:** Gemma 4 Good · Special Track: **Unsloth** ($10K)

Run on Kaggle with **GPU T4 × 2** enabled. Runtime: ~45–90 min for 300 steps.

Produces a LoRA adapter that makes Gemma 4 better at:
- Hindi/English code-mixed rural symptom descriptions
- ASHA-friendly tone (simple, action-oriented, non-clinical jargon)
- Rural-context red-flag recognition (paani-wale dast, bachcha sust, etc.)

## 1. Install dependencies

In [ ]:
!pip install -qU unsloth trl transformers datasets bitsandbytes accelerate peft

## 2. Generate the dataset

In [ ]:
# Either: clone the repo and run prepare_dataset.py
# Or: paste the dataset preparation logic inline
!git clone https://github.com/YOUR_USERNAME/aarogya.git /kaggle/working/aarogya
%cd /kaggle/working/aarogya
!python finetuning/prepare_dataset.py

## 3. Load Gemma 4 with Unsloth + QLoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/gemma-4-4b-it',
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

## 4. Train

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

train_ds = load_dataset('json', data_files='finetuning/data/train.jsonl', split='train')
val_ds   = load_dataset('json', data_files='finetuning/data/val.jsonl',   split='train')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=300,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=50,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir='/kaggle/working/aarogya-gemma4-4b',
        report_to='none',
    ),
)
trainer.train()

## 5. Save LoRA adapter + merged GGUF for Ollama

In [ ]:
# LoRA adapter only (small — ~50 MB)
model.save_pretrained('/kaggle/working/aarogya-gemma4-lora')
tokenizer.save_pretrained('/kaggle/working/aarogya-gemma4-lora')

# Optional: merge & export GGUF for Ollama edge inference
model.save_pretrained_gguf('/kaggle/working/aarogya-gemma4-gguf', tokenizer, quantization_method='q4_k_m')

## 6. Run evaluation (ablation: base vs fine-tuned)

In [ ]:
!python finetuning/evaluate.py --base unsloth/gemma-4-4b-it --finetuned /kaggle/working/aarogya-gemma4-lora